# Multi-Strategy Portfolio

Build and backtest a portfolio with multiple independent trading strategies, each with isolated capital allocations.

**What you'll learn:**
- How to create a multi-strategy portfolio using `PortfolioAllocator`
- Strategy isolation and capital allocation
- Per-strategy performance tracking
- Portfolio-level metrics and correlation analysis

**Prerequisites:**
- Basic understanding of `run_algorithm()` ([Notebook 01](01_basic_strategy.ipynb))
- Familiarity with strategy structure ([Execution Methods Guide](../../guides/execution-methods.md))

**Note**: This is different from Notebook 08 (Portfolio Construction), which shows a single strategy trading multiple assets. Here, we run **multiple independent strategies** simultaneously.

In [ ]:
from decimal import Decimal
from rustybt import TradingAlgorithm
from rustybt.analytics import setup_notebook
from rustybt.api import order_target_percent, symbol
from rustybt.portfolio.allocator import PortfolioAllocator
from rustybt.utils.run_algo import run_algorithm
import pandas as pd
import matplotlib.pyplot as plt

setup_notebook()

## Define Individual Strategies

We'll create three different strategies:
1. **Momentum Strategy**: Buys stocks with positive momentum
2. **Mean Reversion Strategy**: Buys oversold stocks expecting reversion
3. **Trend Following Strategy**: Follows long-term trends

Each strategy will have **isolated capital** and **independent positions**.

In [ ]:
class MomentumStrategy:
    """Buys stocks with positive momentum."""

    def __init__(self, lookback=20):
        self.lookback = lookback
        # Tech stocks
        self.assets = [symbol(s) for s in ['AAPL', 'MSFT', 'GOOGL']]

    def handle_data(self, context, data, ledger):
        """Execute momentum strategy on this strategy's ledger."""
        for asset in self.assets:
            prices = data.history(asset, 'price', self.lookback, '1d')
            if len(prices) < self.lookback:
                continue

            # Calculate momentum
            momentum = (prices[-1] - prices[0]) / prices[0]

            # Trade on signal (uses THIS strategy's capital only)
            if momentum > 0.02:  # Strong momentum
                order_target_percent(asset, 0.33, ledger=ledger)
            elif momentum < -0.02:  # Negative momentum
                order_target_percent(asset, 0, ledger=ledger)


class MeanReversionStrategy:
    """Buys oversold stocks expecting reversion to mean."""

    def __init__(self, lookback=10):
        self.lookback = lookback
        # Defensive assets
        self.assets = [symbol(s) for s in ['TLT', 'GLD', 'VNQ']]

    def handle_data(self, context, data, ledger):
        """Execute mean reversion strategy."""
        for asset in self.assets:
            prices = data.history(asset, 'price', self.lookback, '1d')
            if len(prices) < self.lookback:
                continue

            # Calculate deviation from mean
            mean_price = prices.mean()
            deviation = (prices[-1] - mean_price) / mean_price

            # Trade on reversion signal
            if deviation < -0.03:  # Oversold
                order_target_percent(asset, 0.33, ledger=ledger)
            elif deviation > 0.03:  # Overbought
                order_target_percent(asset, 0, ledger=ledger)


class TrendFollowingStrategy:
    """Follows long-term trends using moving average crossover."""

    def __init__(self, short_window=50, long_window=200):
        self.short_window = short_window
        self.long_window = long_window
        # Broad market ETF
        self.asset = symbol('SPY')

    def handle_data(self, context, data, ledger):
        """Execute trend following strategy."""
        prices = data.history(self.asset, 'price', self.long_window, '1d')
        if len(prices) < self.long_window:
            return

        # Calculate moving averages
        short_ma = prices[-self.short_window:].mean()
        long_ma = prices.mean()

        # Trade on crossover
        if short_ma > long_ma:  # Uptrend
            order_target_percent(self.asset, 1.0, ledger=ledger)
        else:  # Downtrend
            order_target_percent(self.asset, 0, ledger=ledger)

## Create Multi-Strategy Portfolio

Now we'll use `PortfolioAllocator` to manage all three strategies with different capital allocations:
- Momentum: 40% ($400K)
- Mean Reversion: 35% ($350K)
- Trend Following: 25% ($250K)

In [ ]:
def initialize(context):
    """Initialize multi-strategy portfolio."""
    # Create portfolio allocator with total capital
    context.portfolio_allocator = PortfolioAllocator(
        total_capital=Decimal("1000000"),  # $1M total
        name="Multi-Strategy Portfolio"
    )

    # Add Strategy 1: Momentum (40% allocation)
    context.momentum = MomentumStrategy(lookback=20)
    context.portfolio_allocator.add_strategy(
        strategy_id="momentum",
        strategy=context.momentum,
        allocation_pct=Decimal("0.40"),  # 40% = $400K
        metadata={"type": "momentum", "lookback": 20}
    )

    # Add Strategy 2: Mean Reversion (35% allocation)
    context.mean_rev = MeanReversionStrategy(lookback=10)
    context.portfolio_allocator.add_strategy(
        strategy_id="mean_reversion",
        strategy=context.mean_rev,
        allocation_pct=Decimal("0.35"),  # 35% = $350K
        metadata={"type": "mean_reversion", "lookback": 10}
    )

    # Add Strategy 3: Trend Following (25% allocation)
    context.trend = TrendFollowingStrategy(short_window=50, long_window=200)
    context.portfolio_allocator.add_strategy(
        strategy_id="trend_following",
        strategy=context.trend,
        allocation_pct=Decimal("0.25"),  # 25% = $250K
        metadata={"type": "trend_following"}
    )

    print(f"\n{'='*60}")
    print("Multi-Strategy Portfolio Initialized")
    print(f"{'='*60}")
    print(f"Total Capital: ${context.portfolio_allocator.total_capital:,}")
    print(f"Strategies: {len(context.portfolio_allocator.strategies)}")
    for sid, allocation in context.portfolio_allocator.strategies.items():
        capital = allocation.allocated_capital
        pct = (capital / context.portfolio_allocator.total_capital) * 100
        print(f"  - {sid}: ${capital:,} ({pct:.1f}%)")
    print(f"{'='*60}\n")


def handle_data(context, data):
    """Execute all strategies on each bar."""
    # Execute all strategies synchronously (bar-by-bar)
    context.portfolio_allocator.execute_bar(
        timestamp=context.datetime,
        data=data
    )


def analyze(context, perf):
    """Analyze per-strategy and portfolio performance."""
    allocator = context.portfolio_allocator

    print(f"\n{'='*60}")
    print("Per-Strategy Performance")
    print(f"{'='*60}")

    for strategy_id in allocator.strategies:
        metrics = allocator.get_strategy_metrics(strategy_id)
        print(f"\n{strategy_id.upper()}:")
        print(f"  Return:     {metrics.total_return:.2%}")
        print(f"  Sharpe:     {metrics.sharpe_ratio:.2f}")
        print(f"  Max DD:     {metrics.max_drawdown:.2%}")
        print(f"  Volatility: {metrics.volatility:.2%}")
        print(f"  Win Rate:   {metrics.win_rate:.2%}")

    # Portfolio-level metrics
    portfolio_metrics = allocator.get_portfolio_metrics()
    print(f"\n{'='*60}")
    print("Portfolio-Level Performance (Combined)")
    print(f"{'='*60}")
    print(f"  Return:     {portfolio_metrics.total_return:.2%}")
    print(f"  Sharpe:     {portfolio_metrics.sharpe_ratio:.2f}")
    print(f"  Max DD:     {portfolio_metrics.max_drawdown:.2%}")
    print(f"  Volatility: {portfolio_metrics.volatility:.2%}")
    print(f"{'='*60}")

    # Strategy correlation
    correlation = allocator.get_correlation_matrix()
    print(f"\nStrategy Correlation Matrix:")
    print(correlation)
    print()

## Run the Backtest

Let's run the multi-strategy portfolio from 2020 to 2023:

In [ ]:
result = run_algorithm(
    initialize=initialize,
    handle_data=handle_data,
    analyze=analyze,
    bundle='yfinance-profiling',
    start=pd.Timestamp('2020-01-01', tz='UTC'),
    end=pd.Timestamp('2023-12-31', tz='UTC'),
    capital_base=1000000,  # $1M
    data_frequency='daily'
)

## Visualize Results

In [ ]:
# Plot portfolio value over time
fig, ax = plt.subplots(figsize=(12, 6))
result['portfolio_value'].plot(ax=ax, label='Multi-Strategy Portfolio')
ax.set_title('Multi-Strategy Portfolio Value Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot returns distribution
fig, ax = plt.subplots(figsize=(10, 6))
result['returns'].hist(bins=50, ax=ax, alpha=0.7, edgecolor='black')
ax.set_title('Daily Returns Distribution', fontsize=14)
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.axvline(result['returns'].mean(), color='red', linestyle='--', label=f"Mean: {result['returns'].mean():.4f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print(f"\n{'='*60}")
print("Multi-Strategy Portfolio Summary")
print(f"{'='*60}")
print(f"Total Return:       {result['returns'].iloc[-1]:.2%}")
print(f"Sharpe Ratio:       {result['sharpe']:.2f}")
print(f"Max Drawdown:       {result['max_drawdown']:.2%}")
print(f"Volatility:         {result['algo_volatility']:.2%}")
print(f"Final Value:        ${result['portfolio_value'].iloc[-1]:,.2f}")
print(f"Starting Value:     ${result['portfolio_value'].iloc[0]:,.2f}")
print(f"Profit:             ${result['portfolio_value'].iloc[-1] - result['portfolio_value'].iloc[0]:,.2f}")
print(f"{'='*60}\n")

## Key Takeaways

1. **Strategy Isolation**: Each strategy has its own ledger and capital - they operate independently
2. **Diversification**: Multiple uncorrelated strategies reduce overall portfolio volatility
3. **Risk-Adjusted Returns**: Multi-strategy portfolios often achieve higher Sharpe ratios than single strategies
4. **Performance Tracking**: `PortfolioAllocator` tracks both per-strategy and portfolio-level metrics

## Next Steps

- Try different allocation percentages
- Experiment with dynamic rebalancing (see [Multi-Strategy Guide](../../guides/multi-strategy-portfolio-guide.md))
- Add more strategies with different asset classes
- Explore order aggregation to reduce transaction costs

## Related Documentation

- [Multi-Strategy Portfolio Guide](../../guides/multi-strategy-portfolio-guide.md) - Complete guide with advanced features
- [Portfolio Allocation API](../../api/portfolio-management/allocation-multistrategy.md) - Full API reference
- [Order Aggregation](../../api/portfolio-management/order-aggregation.md) - Reduce commissions across strategies
- [Notebook 08: Portfolio Construction](08_portfolio_construction.ipynb) - Single-strategy multi-asset example